# 115_paper_figures_2_14

Unified notebook for the **new-paper** figure set (Figure 2..14) using current artifacts and trained runs.

Existing legacy figure notebooks are kept unchanged.

In [ ]:
import os, sys, pathlib, json
import numpy as np
import matplotlib.pyplot as plt
import torch


def _find_project_root():
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src").is_dir():
            return p
    cand = pathlib.Path("/content/econml")
    if (cand / "src").is_dir():
        return cand
    raise RuntimeError("Could not find project root containing src/.")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ModelParams
from src.io_utils import load_json
from src.metrics import output_gap_from_consumption
from src.table2_builder import _load_run_dir, _load_net_from_run, _load_sim_paths
from src.steady_states import solve_flexprice_sss, solve_taylor_sss, solve_efficient_sss
from src.sss_from_policy import switching_policy_sss_by_regime_from_policy
from src.experiments import DeterministicPathSpec, simulate_deterministic_path

ARTIFACTS_ROOT = str(PROJECT_ROOT / "artifacts")
COMPUTE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Compute device:", COMPUTE_DEVICE)

ann = lambda x: 400.0 * np.asarray(x)


def _parse_dtype(s):
    if s is None:
        return torch.float64
    s = str(s)
    if "float32" in s:
        return torch.float32
    if "float64" in s:
        return torch.float64
    return torch.float64


def load_params_from_run(run_dir: str, *, device: str | None = None) -> ModelParams:
    cfg = load_json(os.path.join(run_dir, "config.json"))
    p = cfg.get("params", {})
    keep = {k:v for k,v in p.items() if k in ModelParams.__dataclass_fields__}
    keep["device"] = device or COMPUTE_DEVICE
    keep["dtype"] = _parse_dtype(p.get("dtype", "float64"))
    return ModelParams(**keep).to_torch()


def copy_params(p: ModelParams, **overrides) -> ModelParams:
    d = {k:getattr(p,k) for k in ModelParams.__dataclass_fields__.keys()}
    d.update(overrides)
    return ModelParams(**d).to_torch()


def load_policy(policy: str):
    run_dir = _load_run_dir(ARTIFACTS_ROOT, policy, use_selected=True)
    params = load_params_from_run(run_dir, device=COMPUTE_DEVICE)
    net = _load_net_from_run(run_dir, params, policy)
    sim = None
    try:
        sim = _load_sim_paths(run_dir)
    except Exception:
        pass
    return run_dir, params, net, sim


def make_rbar(params: ModelParams) -> torch.Tensor:
    flex = solve_flexprice_sss(params)
    return torch.tensor([flex.by_regime[0]["r_star"], flex.by_regime[1]["r_star"]], device=params.device, dtype=params.dtype)


def sss_by_regime(params: ModelParams, policy: str, net):
    if policy in ("mod_taylor", "mod_taylor_zlb"):
        rbar = make_rbar(params)
    else:
        rbar = None
    return switching_policy_sss_by_regime_from_policy(params, net, policy=policy, rbar_by_regime=rbar).by_regime


def x0_from_sss(policy: str, sss: dict, regime: int, *, dtype=torch.float32):
    b = sss[int(regime)]
    if policy == "commitment":
        return torch.tensor([[float(b["Delta_prev"]), float(b["logA"]), float(b["loggtilde"]), float(b["xi"]), float(regime),
                              float(b["vartheta_prev"]), float(b["varrho_prev"]), float(b.get("c_prev", b["c"]))]], dtype=dtype)
    if policy == "commitment_zlb":
        return torch.tensor([[float(b["Delta_prev"]), float(b["logA"]), float(b["loggtilde"]), float(b["xi"]), float(regime),
                              float(b["vartheta_prev"]), float(b["varrho_prev"]), float(b.get("c_prev", b["c"])),
                              float(b.get("i_nom_prev", b.get("i_nom", 0.0))), float(b.get("varphi_prev", b.get("varphi", 0.0)))]], dtype=dtype)
    return torch.tensor([[float(b["Delta_prev"]), float(b["logA"]), float(b["loggtilde"]), float(b["xi"]), float(regime)]], dtype=dtype)


def ensure_same_calibration(params_a: ModelParams, params_b: ModelParams, *, atol: float = 1e-12):
    for k in ModelParams.__dataclass_fields__.keys():
        if k in ("device", "dtype"):
            continue
        va = float(getattr(params_a, k))
        vb = float(getattr(params_b, k))
        if abs(va - vb) > atol:
            raise RuntimeError(f"Calibration mismatch on {k}: {va} vs {vb}")


def series_from_sim(sim: dict, params: ModelParams):
    s = np.asarray(sim["s"]).reshape(-1).astype(np.int64)
    pi = np.asarray(sim["pi"]).reshape(-1)
    i_nom = np.asarray(sim["i"]).reshape(-1)
    x_gap = output_gap_from_consumption(sim, solve_efficient_sss(params), params=params, time_varying=True)
    r = ((1.0 + i_nom[:-1]) / (1.0 + pi[1:])) - 1.0
    return {
        "s": s,
        "pi": pi,
        "i": i_nom,
        "x": np.asarray(x_gap).reshape(-1),
        "r": np.asarray(r).reshape(-1),
        "s_r": s[:-1],
        "Delta": np.asarray(sim["Delta"]).reshape(-1),
    }


## Figure 2: Ergodic distribution (Taylor vs Modified Taylor)

In [ ]:
run_t, p_t, net_t, sim_t = load_policy("taylor")
run_m, p_m, net_m, sim_m = load_policy("mod_taylor")
ensure_same_calibration(p_t, p_m)
if sim_t is None or sim_m is None:
    raise RuntimeError("Figure 2 requires sim_paths.npz for taylor and mod_taylor runs")

ser_t = series_from_sim(sim_t, p_t)
ser_m = series_from_sim(sim_m, p_t)

fig, ax = plt.subplots(2, 2, figsize=(12, 8))

ax[0,0].hist(ann(ser_t["pi"])[ser_t["s"]==0], bins=60, alpha=0.45, label="Taylor normal", color="tab:blue")
ax[0,0].hist(ann(ser_t["pi"])[ser_t["s"]==1], bins=60, alpha=0.45, label="Taylor bad", color="tab:orange")
ax[0,0].hist(ann(ser_m["pi"])[ser_m["s"]==0], bins=60, alpha=0.45, label="Mod Taylor normal", color="tab:green")
ax[0,0].hist(ann(ser_m["pi"])[ser_m["s"]==1], bins=60, alpha=0.45, label="Mod Taylor bad", color="tab:red")
ax[0,0].set_title("Figure 2a: Inflation")
ax[0,0].set_xlabel("Annualized percent")
ax[0,0].legend(fontsize=8)

ax[0,1].hist(100*ser_t["x"][ser_t["s"]==0], bins=60, alpha=0.45, color="tab:blue")
ax[0,1].hist(100*ser_t["x"][ser_t["s"]==1], bins=60, alpha=0.45, color="tab:orange")
ax[0,1].hist(100*ser_m["x"][ser_m["s"]==0], bins=60, alpha=0.45, color="tab:green")
ax[0,1].hist(100*ser_m["x"][ser_m["s"]==1], bins=60, alpha=0.45, color="tab:red")
ax[0,1].set_title("Figure 2b: Output gap")
ax[0,1].set_xlabel("Percent")

ax[1,0].hist(ann(ser_t["i"])[ser_t["s"]==0], bins=60, alpha=0.45, color="tab:blue")
ax[1,0].hist(ann(ser_t["i"])[ser_t["s"]==1], bins=60, alpha=0.45, color="tab:orange")
ax[1,0].hist(ann(ser_m["i"])[ser_m["s"]==0], bins=60, alpha=0.45, color="tab:green")
ax[1,0].hist(ann(ser_m["i"])[ser_m["s"]==1], bins=60, alpha=0.45, color="tab:red")
ax[1,0].set_title("Figure 2c: Nominal rate")
ax[1,0].set_xlabel("Annualized percent")

ax[1,1].hist(ann(ser_t["r"])[ser_t["s_r"]==0], bins=60, alpha=0.45, color="tab:blue")
ax[1,1].hist(ann(ser_t["r"])[ser_t["s_r"]==1], bins=60, alpha=0.45, color="tab:orange")
ax[1,1].hist(ann(ser_m["r"])[ser_m["s_r"]==0], bins=60, alpha=0.45, color="tab:green")
ax[1,1].hist(ann(ser_m["r"])[ser_m["s_r"]==1], bins=60, alpha=0.45, color="tab:red")
ax[1,1].set_title("Figure 2d: Real rate")
ax[1,1].set_xlabel("Annualized percent")

plt.tight_layout(); plt.show()


## Figure 3: Regime-change transition (Taylor vs Modified Taylor)

In [ ]:
sss_t = sss_by_regime(p_t, "taylor", net_t)
sss_m = sss_by_regime(p_m, "mod_taylor", net_m)

x0_t = x0_from_sss("taylor", sss_t, 0)
x0_m = x0_from_sss("mod_taylor", sss_m, 0)

pre, n_post = 5, 20
T = n_post + 1
spec = DeterministicPathSpec(T=T, epsA=0.0, epsg=0.0, epst=0.0, regime_path=[0] + [1]*T)

path_t = simulate_deterministic_path(p_t, "taylor", net_t, x0=x0_t, spec=spec, compute_implied_i=True)
path_m = simulate_deterministic_path(p_m, "mod_taylor", net_m, x0=x0_m, spec=spec, rbar_by_regime=make_rbar(p_m), compute_implied_i=True)

pi_t = path_t["pi"][:,0]; pi_m = path_m["pi"][:,0]
i_t = path_t["i"][:,0]; i_m = path_m["i"][:,0]
Delta_t = path_t["Delta"][:,0]; Delta_m = path_m["Delta"][:,0]

eff = solve_efficient_sss(p_t)
c_hat=float(eff["c_hat"])
x_t = np.log(path_t["c"][:,0]) - np.log(c_hat)
x_m = np.log(path_m["c"][:,0]) - np.log(c_hat)
r_t = ((1+i_t[:-1])/(1+pi_t[1:]))-1
r_m = ((1+i_m[:-1])/(1+pi_m[1:]))-1

pi_t = np.concatenate([np.full(pre, pi_t[0]), pi_t[1:1+n_post]])
pi_m = np.concatenate([np.full(pre, pi_m[0]), pi_m[1:1+n_post]])
x_t = np.concatenate([np.full(pre, x_t[0]), x_t[1:1+n_post]])
x_m = np.concatenate([np.full(pre, x_m[0]), x_m[1:1+n_post]])
r_t = np.concatenate([np.full(pre, r_t[1]), r_t[1:1+n_post]])
r_m = np.concatenate([np.full(pre, r_m[1]), r_m[1:1+n_post]])
D_t = np.concatenate([np.full(pre, Delta_t[0]), Delta_t[1:1+n_post]])
D_m = np.concatenate([np.full(pre, Delta_m[0]), Delta_m[1:1+n_post]])

t=np.arange(-pre,n_post)
fig,ax=plt.subplots(2,2,figsize=(12,8))
ax[0,0].plot(t,ann(pi_t),label='Taylor')
ax[0,0].plot(t,ann(pi_m),'--',label='Mod. Taylor')
ax[0,0].set_title('Figure 3a: Inflation'); ax[0,0].legend(); ax[0,0].axhline(0,color='k',lw=1)
ax[0,1].plot(t,100*x_t,label='Taylor'); ax[0,1].plot(t,100*x_m,'--',label='Mod. Taylor')
ax[0,1].set_title('Figure 3b: Output gap'); ax[0,1].axhline(0,color='k',lw=1)
ax[1,0].plot(t,ann(r_t),label='Taylor'); ax[1,0].plot(t,ann(r_m),'--',label='Mod. Taylor')
ax[1,0].set_title('Figure 3c: Real rate'); ax[1,0].axhline(0,color='k',lw=1)
ax[1,1].plot(t,D_t,label='Taylor'); ax[1,1].plot(t,D_m,'--',label='Mod. Taylor')
ax[1,1].set_title('Figure 3d: Price dispersion')
for a in ax.ravel(): a.set_xlabel('Time in quarters')
plt.tight_layout(); plt.show()


## Figure 4: Sensitivity to bad-regime duration (Taylor rule)

In [ ]:
params0 = ModelParams(device=COMPUTE_DEVICE, dtype=torch.float64).to_torch()
grid_p21 = np.linspace(0.02, 0.98, 40)
dur_bad = 1.0 / grid_p21

pi_normal = []; r_normal = []; pi_bad_cf = []; r_bad_cf = []
for p21 in grid_p21:
    p_base = copy_params(params0, p12=float(params0.p12), p21=float(p21))
    flex_b = solve_flexprice_sss(p_base)
    t_b = solve_taylor_sss(p_base, flex_b)
    pi_normal.append(float(t_b.by_regime[0]['pi']))
    r_normal.append(float(t_b.by_regime[0]['r']))

    p_cf = copy_params(params0, p12=1.0, p21=float(p21))
    flex_c = solve_flexprice_sss(p_cf)
    t_c = solve_taylor_sss(p_cf, flex_c)
    pi_bad_cf.append(float(t_c.by_regime[1]['pi']))
    r_bad_cf.append(float(t_c.by_regime[1]['r']))

eff = solve_efficient_sss(params0)
x = dur_bad[np.argsort(dur_bad)]
idx = np.argsort(dur_bad)

fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(x, ann(np.array(pi_normal)[idx]), label='Normal SSS (baseline)')
ax[0].plot(x, ann(np.array(pi_bad_cf)[idx]), '--', label='Bad SSS (p12->1)')
ax[0].axhline(0.0, color='gray', ls=':')
ax[0].set_title('Figure 4a: SSS inflation sensitivity')
ax[0].set_xlabel('Average bad-times duration (quarters)')
ax[0].set_ylabel('Annualized percent')
ax[0].legend()

ax[1].plot(x, ann(np.array(r_normal)[idx]), label='Normal SSS (baseline)')
ax[1].plot(x, ann(np.array(r_bad_cf)[idx]), '--', label='Bad SSS (p12->1)')
ax[1].axhline(ann(float(eff['r_hat'])), color='gray', ls=':')
ax[1].set_title('Figure 4b: SSS real-rate sensitivity')
ax[1].set_xlabel('Average bad-times duration (quarters)')
ax[1].set_ylabel('Annualized percent')
ax[1].legend()

plt.tight_layout(); plt.show()


## Figure 5/6/10/14: Ergodic distributions for policy pairs

In [ ]:
def plot_ergodic_pair(sim_a, sim_b, params, title_prefix, labels):
    sa = series_from_sim(sim_a, params)
    sb = series_from_sim(sim_b, params)
    fig,ax=plt.subplots(2,2,figsize=(12,8))

    ax[0,0].hist(ann(sa['pi'])[sa['s']==0], bins=60, alpha=0.45, label=f"{labels[0]} normal", color='tab:blue')
    ax[0,0].hist(ann(sa['pi'])[sa['s']==1], bins=60, alpha=0.45, label=f"{labels[0]} bad", color='tab:orange')
    ax[0,0].hist(ann(sb['pi'])[sb['s']==0], bins=60, alpha=0.45, label=f"{labels[1]} normal", color='tab:green')
    ax[0,0].hist(ann(sb['pi'])[sb['s']==1], bins=60, alpha=0.45, label=f"{labels[1]} bad", color='tab:red')
    ax[0,0].set_title(f"{title_prefix}a: Inflation")
    ax[0,0].legend(fontsize=8)

    ax[0,1].hist(100*sa['x'][sa['s']==0], bins=60, alpha=0.45, color='tab:blue')
    ax[0,1].hist(100*sa['x'][sa['s']==1], bins=60, alpha=0.45, color='tab:orange')
    ax[0,1].hist(100*sb['x'][sb['s']==0], bins=60, alpha=0.45, color='tab:green')
    ax[0,1].hist(100*sb['x'][sb['s']==1], bins=60, alpha=0.45, color='tab:red')
    ax[0,1].set_title(f"{title_prefix}b: Output gap")

    ax[1,0].hist(ann(sa['r'])[sa['s_r']==0], bins=60, alpha=0.45, color='tab:blue')
    ax[1,0].hist(ann(sa['r'])[sa['s_r']==1], bins=60, alpha=0.45, color='tab:orange')
    ax[1,0].hist(ann(sb['r'])[sb['s_r']==0], bins=60, alpha=0.45, color='tab:green')
    ax[1,0].hist(ann(sb['r'])[sb['s_r']==1], bins=60, alpha=0.45, color='tab:red')
    ax[1,0].set_title(f"{title_prefix}c: Real rate")

    ax[1,1].hist(sa['Delta'][sa['s']==0], bins=60, alpha=0.45, color='tab:blue')
    ax[1,1].hist(sa['Delta'][sa['s']==1], bins=60, alpha=0.45, color='tab:orange')
    ax[1,1].hist(sb['Delta'][sb['s']==0], bins=60, alpha=0.45, color='tab:green')
    ax[1,1].hist(sb['Delta'][sb['s']==1], bins=60, alpha=0.45, color='tab:red')
    ax[1,1].set_title(f"{title_prefix}d: Price dispersion")

    for a in ax.ravel():
        a.set_xlabel('model units')
    plt.tight_layout(); plt.show()

# Figure 5 (discretion, single-policy view via duplicated pair)
_, p_d, net_d, sim_d = load_policy('discretion')
if sim_d is None: raise RuntimeError('Figure 5 requires discretion sim_paths.npz')
plot_ergodic_pair(sim_d, sim_d, p_d, 'Figure 5', ('Discretion','Discretion'))

# Figure 6 (commitment, single-policy view via duplicated pair)
_, p_c, net_c, sim_c = load_policy('commitment')
if sim_c is None: raise RuntimeError('Figure 6 requires commitment sim_paths.npz')
plot_ergodic_pair(sim_c, sim_c, p_c, 'Figure 6', ('Commitment','Commitment'))

# Figure 10 (Taylor ZLB vs Modified Taylor ZLB)
_, p_tz, net_tz, sim_tz = load_policy('taylor_zlb')
_, p_mz, net_mz, sim_mz = load_policy('mod_taylor_zlb')
if sim_tz is None or sim_mz is None: raise RuntimeError('Figure 10 requires taylor_zlb and mod_taylor_zlb sim_paths.npz')
ensure_same_calibration(p_tz, p_mz)
plot_ergodic_pair(sim_tz, sim_mz, p_tz, 'Figure 10', ('Taylor ZLB','Mod Taylor ZLB'))

# Figure 14 (Discretion ZLB vs Commitment ZLB)
_, p_dz, net_dz, sim_dz = load_policy('discretion_zlb')
_, p_cz, net_cz, sim_cz = load_policy('commitment_zlb')
if sim_dz is None or sim_cz is None: raise RuntimeError('Figure 14 requires discretion_zlb and commitment_zlb sim_paths.npz')
ensure_same_calibration(p_dz, p_cz)
plot_ergodic_pair(sim_dz, sim_cz, p_dz, 'Figure 14', ('Discretion ZLB','Commitment ZLB'))


## Figure 8/11: Regime-change transitions for optimal policy pairs

In [ ]:
# Figure 8: commitment vs discretion
sss_c = sss_by_regime(p_c, 'commitment', net_c)
sss_d = sss_by_regime(p_d, 'discretion', net_d)
x0_c = x0_from_sss('commitment', sss_c, 0)
x0_d = x0_from_sss('discretion', sss_d, 0)

pre, n_post = 5, 20
T = n_post + 1
spec_sw = DeterministicPathSpec(T=T, epsA=0.0, epsg=0.0, epst=0.0, regime_path=[0] + [1]*T)

path_c = simulate_deterministic_path(p_c, 'commitment', net_c, x0=x0_c, spec=spec_sw, compute_implied_i=True)
path_d = simulate_deterministic_path(p_d, 'discretion', net_d, x0=x0_d, spec=spec_sw, compute_implied_i=True)

def _plot_transition_pair(path_a, path_b, params, fig_prefix, labels):
    eff = solve_efficient_sss(params)
    c_hat = float(eff['c_hat'])
    pi_a, pi_b = path_a['pi'][:,0], path_b['pi'][:,0]
    i_a, i_b = path_a['i'][:,0], path_b['i'][:,0]
    x_a = np.log(path_a['c'][:,0]) - np.log(c_hat)
    x_b = np.log(path_b['c'][:,0]) - np.log(c_hat)
    r_a = ((1+i_a[:-1])/(1+pi_a[1:]))-1
    r_b = ((1+i_b[:-1])/(1+pi_b[1:]))-1
    D_a, D_b = path_a['Delta'][:,0], path_b['Delta'][:,0]

    pi_a = np.concatenate([np.full(pre, pi_a[0]), pi_a[1:1+n_post]])
    pi_b = np.concatenate([np.full(pre, pi_b[0]), pi_b[1:1+n_post]])
    x_a = np.concatenate([np.full(pre, x_a[0]), x_a[1:1+n_post]])
    x_b = np.concatenate([np.full(pre, x_b[0]), x_b[1:1+n_post]])
    r_a = np.concatenate([np.full(pre, r_a[1]), r_a[1:1+n_post]])
    r_b = np.concatenate([np.full(pre, r_b[1]), r_b[1:1+n_post]])
    D_a = np.concatenate([np.full(pre, D_a[0]), D_a[1:1+n_post]])
    D_b = np.concatenate([np.full(pre, D_b[0]), D_b[1:1+n_post]])

    t=np.arange(-pre,n_post)
    fig,ax=plt.subplots(2,2,figsize=(12,8))
    ax[0,0].plot(t,ann(pi_a),label=labels[0]); ax[0,0].plot(t,ann(pi_b),'--',label=labels[1]); ax[0,0].axhline(0,color='k',lw=1)
    ax[0,0].set_title(f'{fig_prefix}a: Inflation')
    ax[0,1].plot(t,100*x_a,label=labels[0]); ax[0,1].plot(t,100*x_b,'--',label=labels[1]); ax[0,1].axhline(0,color='k',lw=1)
    ax[0,1].set_title(f'{fig_prefix}b: Output gap')
    ax[1,0].plot(t,ann(r_a),label=labels[0]); ax[1,0].plot(t,ann(r_b),'--',label=labels[1]); ax[1,0].axhline(0,color='k',lw=1)
    ax[1,0].set_title(f'{fig_prefix}c: Real rate')
    ax[1,1].plot(t,D_a,label=labels[0]); ax[1,1].plot(t,D_b,'--',label=labels[1])
    ax[1,1].set_title(f'{fig_prefix}d: Price dispersion')
    for a in ax.ravel():
        a.set_xlabel('Time in quarters')
    ax[0,0].legend()
    plt.tight_layout(); plt.show()

_plot_transition_pair(path_c, path_d, p_c, 'Figure 8', ('Commitment','Discretion'))

# Figure 11: commitment (no ZLB) vs commitment_zlb
sss_cz = sss_by_regime(p_cz, 'commitment_zlb', net_cz)
x0_cz = x0_from_sss('commitment_zlb', sss_cz, 0)
path_cz = simulate_deterministic_path(p_cz, 'commitment_zlb', net_cz, x0=x0_cz, spec=spec_sw, compute_implied_i=True)
_plot_transition_pair(path_c, path_cz, p_c, 'Figure 11', ('Commitment','Commitment ZLB'))


## Figure 7/9/12/13: Commitment impulse and price-level dynamics

In [ ]:
# Common commitment baseline
xi_jump = float(p_c.sigma_tau) * 3.0
T = 24

# Figure 7: transitory cost-push shock in bad regime, commitment vs modified Taylor
sss_m = sss_by_regime(p_m, 'mod_taylor', net_m)
x0_cb = x0_from_sss('commitment', sss_c, 1)
x0_mb = x0_from_sss('mod_taylor', sss_m, 1)
x0_cb[:,3] += xi_jump
x0_mb[:,3] += xi_jump
spec_bad = DeterministicPathSpec(T=T, epsA=0.0, epsg=0.0, epst=0.0, regime_path=[1]*(T+1))
path_cb = simulate_deterministic_path(p_c, 'commitment', net_c, x0=x0_cb, spec=spec_bad, compute_implied_i=True)
path_mb = simulate_deterministic_path(p_m, 'mod_taylor', net_m, x0=x0_mb, spec=spec_bad, rbar_by_regime=make_rbar(p_m), compute_implied_i=True)


def _plot_impulse(path_a, path_b, params, fig_prefix, labels):
    eff = solve_efficient_sss(params)
    c_hat = float(eff['c_hat'])
    pi_a, pi_b = path_a['pi'][:,0], path_b['pi'][:,0]
    i_a, i_b = path_a['i'][:,0], path_b['i'][:,0]
    x_a = np.log(path_a['c'][:,0]) - np.log(c_hat)
    x_b = np.log(path_b['c'][:,0]) - np.log(c_hat)
    r_a = ((1+i_a[:-1])/(1+pi_a[1:]))-1
    r_b = ((1+i_b[:-1])/(1+pi_b[1:]))-1
    P_a = np.cumprod(1.0 + pi_a)
    P_b = np.cumprod(1.0 + pi_b)
    t=np.arange(len(pi_a))
    fig,ax=plt.subplots(2,2,figsize=(12,8))
    ax[0,0].plot(t,ann(pi_a),label=labels[0]); ax[0,0].plot(t,ann(pi_b),'--',label=labels[1]); ax[0,0].axhline(0,color='k',lw=1)
    ax[0,0].set_title(f'{fig_prefix}a: Inflation')
    ax[0,1].plot(t,100*x_a,label=labels[0]); ax[0,1].plot(t,100*x_b,'--',label=labels[1]); ax[0,1].axhline(0,color='k',lw=1)
    ax[0,1].set_title(f'{fig_prefix}b: Output gap')
    ax[1,0].plot(t[:-1],ann(r_a),label=labels[0]); ax[1,0].plot(t[:-1],ann(r_b),'--',label=labels[1]); ax[1,0].axhline(0,color='k',lw=1)
    ax[1,0].set_title(f'{fig_prefix}c: Real rate')
    ax[1,1].plot(t,P_a,label=labels[0]); ax[1,1].plot(t,P_b,'--',label=labels[1])
    ax[1,1].set_title(f'{fig_prefix}d: Price level')
    for a in ax.ravel():
        a.set_xlabel('Time in quarters')
    ax[0,0].legend()
    plt.tight_layout(); plt.show()

_plot_impulse(path_cb, path_mb, p_c, 'Figure 7', ('Commitment','Mod. Taylor'))

# Figure 9: different persistence under commitment (rho_tau baseline vs 0.99)
x0_p = x0_from_sss('commitment', sss_c, 1)
x0_p[:,3] += xi_jump
path_rho_base = simulate_deterministic_path(p_c, 'commitment', net_c, x0=x0_p, spec=spec_bad, compute_implied_i=True)
p_hi = copy_params(p_c, rho_tau=0.99)
path_rho_hi = simulate_deterministic_path(p_hi, 'commitment', net_c, x0=x0_p, spec=spec_bad, compute_implied_i=True)
_plot_impulse(path_rho_base, path_rho_hi, p_c, 'Figure 9', ('rho_tau=baseline','rho_tau=0.99'))

# Figure 12: long-run price level under commitment
if sim_c is None:
    raise RuntimeError('Figure 12 requires commitment sim_paths.npz')
pi_sim = np.asarray(sim_c['pi']).reshape(-1)
P = np.cumprod(1.0 + pi_sim)
T_show = min(2000, len(P))
fig,ax=plt.subplots(1,1,figsize=(11,4))
ax.plot(np.arange(T_show), P[:T_show])
ax.set_title('Figure 12: Price level dynamics under commitment')
ax.set_xlabel('Quarter'); ax.set_ylabel('Price level index')
plt.tight_layout(); plt.show()

# Figure 13: regime-change transition with larger persistent supply shock (eta_bar)
p_big = copy_params(p_c, eta_bar=float(p_c.eta_bar)*1.5)
sss_big = switching_policy_sss_by_regime_from_policy(p_big, net_c, policy='commitment').by_regime
x0_big = x0_from_sss('commitment', sss_big, 0)
spec_sw2 = DeterministicPathSpec(T=21, epsA=0.0, epsg=0.0, epst=0.0, regime_path=[0] + [1]*21)
path_base = simulate_deterministic_path(p_c, 'commitment', net_c, x0=x0_from_sss('commitment', sss_c, 0), spec=spec_sw2, compute_implied_i=True)
path_big = simulate_deterministic_path(p_big, 'commitment', net_c, x0=x0_big, spec=spec_sw2, compute_implied_i=True)

# plot as transition pair
pre, n_post = 5, 20
def _transition_arrays(path, params):
    eff = solve_efficient_sss(params)
    c_hat = float(eff['c_hat'])
    pi = path['pi'][:,0]; i = path['i'][:,0]
    x = np.log(path['c'][:,0]) - np.log(c_hat)
    r = ((1+i[:-1])/(1+pi[1:]))-1
    D = path['Delta'][:,0]
    pi = np.concatenate([np.full(pre, pi[0]), pi[1:1+n_post]])
    x = np.concatenate([np.full(pre, x[0]), x[1:1+n_post]])
    r = np.concatenate([np.full(pre, r[1]), r[1:1+n_post]])
    D = np.concatenate([np.full(pre, D[0]), D[1:1+n_post]])
    return pi,x,r,D
pi0,x0,r0,D0 = _transition_arrays(path_base, p_c)
pi1,x1,r1,D1 = _transition_arrays(path_big, p_big)
t=np.arange(-pre,n_post)
fig,ax=plt.subplots(2,2,figsize=(12,8))
ax[0,0].plot(t,ann(pi0),label='baseline'); ax[0,0].plot(t,ann(pi1),'--',label='larger shock')
ax[0,0].set_title('Figure 13a: Inflation'); ax[0,0].axhline(0,color='k',lw=1)
ax[0,1].plot(t,100*x0,label='baseline'); ax[0,1].plot(t,100*x1,'--',label='larger shock')
ax[0,1].set_title('Figure 13b: Output gap'); ax[0,1].axhline(0,color='k',lw=1)
ax[1,0].plot(t,ann(r0),label='baseline'); ax[1,0].plot(t,ann(r1),'--',label='larger shock')
ax[1,0].set_title('Figure 13c: Real rate'); ax[1,0].axhline(0,color='k',lw=1)
ax[1,1].plot(t,D0,label='baseline'); ax[1,1].plot(t,D1,'--',label='larger shock')
ax[1,1].set_title('Figure 13d: Price dispersion')
for a in ax.ravel(): a.set_xlabel('Time in quarters')
ax[0,0].legend(); plt.tight_layout(); plt.show()


### Notes
- Uses currently selected/latest runs for each policy.
- Figures that require ZLB policies need completed `14..17` training runs.
- If a required run is missing, the corresponding cell raises a clear error.